# Results Robustness and Rescue Checks

This notebook stress-tests whether the PCA residual strategy can be rescued by fixing signal direction, reducing turnover, changing the holding horizon, tightening thresholds, and hardening the factor-neutral implementation.

In [17]:
import itertools
import os
import sys
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pca_residual_mean_reversion.backtest import backtest, backtest_holding_period
from pca_residual_mean_reversion.metrics import perf_metrics
from pca_residual_mean_reversion.portfolio import (
    apply_no_trade_band,
    build_cross_sectional_reversal,
    build_weights_equal,
    compute_factor_exposures,
    neutralize_weights_over_time,
    rebalance_every_n_days,
    smooth_weights,
)

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

PROCESSED = "../data/processed"
TABLES_DIR = "../reports/tables"
FIGURES_DIR = "../reports/figures"
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

MAIN_SUFFIX = "_L126_K10_M20"

L_GRID = [63, 126, 252]
K_GRID = [5]
M_GRID = [20]
C_GRID = [1.0]
COST_GRID = [0, 1, 2, 5, 10]
REBALANCE_GRID = [5, 10, 21]
BAND_GRID = [0.001, 0.0025, 0.005, 0.01]
SMOOTH_GRID = [0.05, 0.1, 0.2, 0.5]
THRESHOLD_GRID = [1.0, 1.25, 1.5, 1.75, 2.0, 2.5]
HOLDING_GRID = [1, 2, 5, 10, 21]
REVERSAL_M_GRID = [1, 5, 10, 20]
RANDOM_SEEDS = 100

2026-05-04 01:45:44.998 | INFO     | pca_residual_mean_reversion.config:<module>:11 - PROJ_ROOT path is: /Users/ashleygarcia/Workspace/pca-residual-mean-reversion


## Helper Functions

In [18]:
def load_zscores(suffix, processed=PROCESSED):
    path = f"{processed}/residual_zscores{suffix}.parquet"
    if not os.path.exists(path):
        return None
    return pd.read_parquet(path)


def load_residuals(suffix, processed=PROCESSED):
    path = f"{processed}/residual_returns{suffix}.parquet"
    if not os.path.exists(path):
        return None
    return pd.read_parquet(path)


def load_log_returns(processed=PROCESSED):
    return pd.read_parquet(f"{processed}/log_returns.parquet")


def load_rolling_loadings(suffix, processed=PROCESSED):
    path = f"{processed}/pca_rolling_loadings{suffix}.pkl"
    if not os.path.exists(path):
        return None
    with open(path, "rb") as fh:
        payload = pickle.load(fh)
    return payload["loadings_by_date"]


def build_weights_quantile(zscores, q=0.10, G=1.0):
    w = pd.DataFrame(0.0, index=zscores.index, columns=zscores.columns)
    for dt, row in zscores.iterrows():
        lo = row.quantile(q)
        hi = row.quantile(1 - q)
        long_mask = row <= lo
        short_mask = row >= hi
        n_long = long_mask.sum()
        n_short = short_mask.sum()
        if n_long > 0:
            w.loc[dt, long_mask] = (G / 2) / n_long
        if n_short > 0:
            w.loc[dt, short_mask] = -(G / 2) / n_short
    return w


def build_random_long_short_weights(log_returns, seed=42, G=1.0):
    rng = np.random.default_rng(seed)
    n = log_returns.shape[1]
    rows = []
    for _ in range(len(log_returns)):
        perm = rng.permutation(n)
        half = n // 2
        w = np.zeros(n)
        w[perm[:half]] = (G / 2) / max(half, 1)
        w[perm[half: half * 2]] = -(G / 2) / max(n - half, 1)
        rows.append(w)
    return pd.DataFrame(rows, index=log_returns.index, columns=log_returns.columns)


def run_backtest_table(weights_map, returns, cost_bps=5, holding_days=1):
    rows = {}
    backtests = {}
    for name, weights in weights_map.items():
        bt = backtest(weights, returns, cost_bps=cost_bps) if holding_days == 1 else backtest_holding_period(weights, returns, holding_days=holding_days, cost_bps=cost_bps)
        backtests[name] = bt
        rows[name] = perf_metrics(bt["net"])
        rows[name]["Mean Turnover"] = bt["turnover"].mean()
    return pd.DataFrame(rows).T, backtests


def format_results(df):
    fmt = {
        "Ann. Return": "{:.2%}",
        "Ann. Vol": "{:.2%}",
        "Sharpe": "{:.2f}",
        "Max Drawdown": "{:.2%}",
        "Calmar": "{:.2f}",
        "Hit Rate": "{:.1%}",
        "N Days": "{:.0f}",
        "Mean Turnover": "{:.3f}",
    }
    return df.style.format({k: v for k, v in fmt.items() if k in df.columns}, na_rep="--")

## 1. Targeted Window Sensitivity

This section is a targeted robustness check across PCA estimation windows, holding the remaining parameters fixed at `K=5`, `M=20`, and threshold `c=1.0`.

In [19]:
log_returns = load_log_returns()
grid_results = {}
missing_specs = []
expected_specs = len(L_GRID) * len(K_GRID) * len(M_GRID) * len(C_GRID)

for L, K, M, c_thresh in itertools.product(L_GRID, K_GRID, M_GRID, C_GRID):
    suffix = f"_L{L}_K{K}_M{M}"
    zs = load_zscores(suffix)
    if zs is None:
        missing_specs.append((L, K, M))
        continue
    shared = zs.columns.intersection(log_returns.columns)
    zs_clean = zs[shared]
    lr_clean = log_returns[shared].reindex(zs_clean.index)
    w = build_weights_equal(zs_clean, c=c_thresh, G=1.0, w_max=0.05)
    bt = backtest(w, lr_clean, cost_bps=5)
    pm = perf_metrics(bt["net"])
    pm["Mean Turnover"] = bt["turnover"].mean()
    grid_results[(L, K, M, c_thresh)] = pm

print(f"Expected rows: {expected_specs}")
print(f"Computed rows: {len(grid_results)}")
print(f"Missing specs: {len(missing_specs)}")
if missing_specs:
    print("Window sensitivity incomplete. Generate the missing specs:")
    print(sorted(missing_specs))
else:
    print("Window sensitivity table is fully populated.")

Specs computed: 3
Missing (L, K, M): 8
[(63, 5, 20), (63, 10, 20), (63, 15, 20), (126, 5, 20), (126, 15, 20), (252, 5, 20), (252, 10, 20), (252, 15, 20)]


In [20]:
if grid_results:
    grid_df = pd.DataFrame(grid_results).T
    grid_df.index.names = ["L", "K", "M", "threshold"]
    window_df = grid_df.reset_index().drop(columns=["K", "M", "threshold"]).set_index("L").sort_index()
    window_df.to_csv(f"{TABLES_DIR}/window_sensitivity{MAIN_SUFFIX}.csv")
    print(window_df[["Sharpe", "Ann. Return", "Mean Turnover"]].to_string())

                       Sharpe  Ann. Return  Mean Turnover
L   K  M  threshold                                      
126 10 20 2.0       -2.586499    -0.237681       1.919450
          1.5       -4.419957    -0.225926       1.855675
          1.0       -6.379310    -0.208704       1.708524


## 2. Signal Direction and Core Rescue Experiments

In [21]:
zs_main = load_zscores(MAIN_SUFFIX)
if zs_main is None:
    raise FileNotFoundError(f"Missing zscores for {MAIN_SUFFIX}. Re-run notebook 2.0.")

lr_main = load_log_returns()
shared = zs_main.columns.intersection(lr_main.columns)
zs_main = zs_main[shared]
lr_main = lr_main[shared].reindex(zs_main.index)

weights_main = build_weights_equal(zs_main, c=1.0, G=1.0, w_max=0.05)
weights_inverted = -weights_main

signal_direction = pd.DataFrame({
    "Main 0bps": perf_metrics(backtest(weights_main, lr_main, cost_bps=0)["net"]),
    "Inverted 0bps": perf_metrics(backtest(weights_inverted, lr_main, cost_bps=0)["net"]),
    "Main 5bps": perf_metrics(backtest(weights_main, lr_main, cost_bps=5)["net"]),
    "Inverted 5bps": perf_metrics(backtest(weights_inverted, lr_main, cost_bps=5)["net"]),
}).T
signal_direction.to_csv(f"{TABLES_DIR}/signal_direction{MAIN_SUFFIX}.csv")
format_results(signal_direction)

,Ann. Return,Ann. Vol,Sharpe,Max Drawdown,Calmar,Hit Rate,N Days
Main 0bps,-1.85%,3.27%,-0.57,-31.16%,-0.06,48.1%,3584
Inverted 0bps,1.78%,3.27%,0.54,-10.35%,0.17,51.9%,3584
Main 5bps,-20.87%,3.27%,-6.38,-96.42%,-0.22,31.8%,3584
Inverted 5bps,-17.94%,3.27%,-5.48,-94.00%,-0.19,34.1%,3584


In [22]:
weekly_weights = {f"Weekly n={n}": rebalance_every_n_days(weights_main, n=n) for n in REBALANCE_GRID}
band_weights = {f"Band {band:.4f}": apply_no_trade_band(weights_main, band=band) for band in BAND_GRID}
smooth_weight_map = {f"Smooth alpha={alpha}": smooth_weights(weights_main, alpha=alpha, gross_target=1.0) for alpha in SMOOTH_GRID}

turnover_variants = {"Daily main": weights_main, **weekly_weights, **band_weights, **smooth_weight_map}
turnover_table, turnover_backtests = run_backtest_table(turnover_variants, lr_main, cost_bps=5)
turnover_table = turnover_table.sort_values("Sharpe", ascending=False)
turnover_table.to_csv(f"{TABLES_DIR}/turnover_controls{MAIN_SUFFIX}.csv")
format_results(turnover_table)

,Ann. Return,Ann. Vol,Sharpe,Max Drawdown,Calmar,Hit Rate,N Days,Mean Turnover
Weekly n=21,-0.74%,3.41%,-0.22,-21.37%,-0.03,50.1%,3584,0.083
Weekly n=10,-1.60%,3.24%,-0.49,-24.23%,-0.07,49.1%,3584,0.176
Weekly n=5,-3.41%,3.23%,-1.05,-41.33%,-0.08,47.6%,3584,0.346
Smooth alpha=0.05,-3.98%,2.41%,-1.65,-45.13%,-0.09,45.6%,3584,0.301
Smooth alpha=0.1,-5.27%,2.43%,-2.17,-54.73%,-0.10,44.4%,3584,0.405
Smooth alpha=0.2,-7.51%,2.43%,-3.09,-67.47%,-0.11,41.1%,3584,0.575
Smooth alpha=0.5,-13.00%,2.54%,-5.11,-86.20%,-0.15,35.1%,3584,1.010
Band 0.0100,-20.72%,3.32%,-6.24,-96.32%,-0.22,32.0%,3584,1.679
Band 0.0050,-20.64%,3.28%,-6.29,-96.27%,-0.21,31.8%,3584,1.692
Band 0.0025,-20.85%,3.28%,-6.36,-96.41%,-0.22,31.8%,3584,1.703


In [23]:
threshold_rows = {}
for c in THRESHOLD_GRID:
    w = build_weights_equal(zs_main, c=c, G=1.0, w_max=0.05)
    bt = backtest(w, lr_main, cost_bps=5)
    threshold_rows[c] = {**perf_metrics(bt["net"]), "Mean Turnover": bt["turnover"].mean()}

threshold_df = pd.DataFrame(threshold_rows).T
threshold_df.index.name = "threshold"
threshold_df.to_csv(f"{TABLES_DIR}/threshold_sweep{MAIN_SUFFIX}.csv")
format_results(threshold_df.sort_index())

,Ann. Return,Ann. Vol,Sharpe,Max Drawdown,Calmar,Hit Rate,N Days,Mean Turnover
threshold,,,,,,,,
1.000000,-20.87%,3.27%,-6.38,-96.42%,-0.22,31.8%,3584,1.709
1.250000,-21.79%,4.06%,-5.36,-96.97%,-0.22,32.6%,3584,1.794
1.500000,-22.59%,5.11%,-4.42,-97.38%,-0.23,35.9%,3584,1.856
1.750000,-22.74%,6.70%,-3.40,-97.46%,-0.23,39.3%,3584,1.895
2.000000,-23.77%,9.19%,-2.59,-97.95%,-0.24,40.5%,3584,1.919
2.500000,-25.50%,13.19%,-1.93,-98.52%,-0.26,42.6%,3584,1.931


In [24]:
holding_rows = {}
for h in HOLDING_GRID:
    bt = backtest_holding_period(weights_main, lr_main, holding_days=h, cost_bps=5)
    holding_rows[h] = {**perf_metrics(bt["net"]), "Mean Turnover": bt["turnover"].mean()}

holding_df = pd.DataFrame(holding_rows).T
holding_df.index.name = "holding_days"
holding_df.to_csv(f"{TABLES_DIR}/holding_period_sweep{MAIN_SUFFIX}.csv")
format_results(holding_df.sort_index())

,Ann. Return,Ann. Vol,Sharpe,Max Drawdown,Calmar,Hit Rate,N Days,Mean Turnover
holding_days,,,,,,,,
1,-20.87%,3.27%,-6.38,-96.42%,-0.22,31.8%,3584,1.709
2,-20.21%,2.32%,-8.70,-95.96%,-0.21,25.3%,3584,1.709
5,-19.56%,1.46%,-13.38,-95.48%,-0.20,15.2%,3584,1.709
10,-19.52%,1.04%,-18.78,-95.44%,-0.20,8.4%,3584,1.709
21,-19.37%,0.72%,-27.07,-95.32%,-0.20,2.6%,3584,1.709


## 3. Transaction Cost Sensitivity

In [25]:
cost_sens = {}
for c_bps in COST_GRID:
    bt = backtest(weights_main, lr_main, cost_bps=c_bps)
    cost_sens[f"{c_bps} bps"] = perf_metrics(bt["net"])

cost_sens_df = pd.DataFrame(cost_sens).T[["Ann. Return", "Ann. Vol", "Sharpe", "Max Drawdown"]]
cost_sens_df.to_csv(f"{TABLES_DIR}/cost_sensitivity{MAIN_SUFFIX}.csv")
print(cost_sens_df.to_string())

        Ann. Return  Ann. Vol     Sharpe  Max Drawdown
0 bps     -0.018527  0.032709  -0.566421     -0.311645
1 bps     -0.059894  0.032709  -1.831135     -0.592358
2 bps     -0.099524  0.032709  -3.042700     -0.775971
5 bps     -0.208704  0.032716  -6.379310     -0.964200
10 bps    -0.362149  0.032745 -11.059690     -0.998329


## 4. Baselines and Better Comparators

In [26]:
w_main = weights_main
bt_main = backtest(w_main, lr_main, cost_bps=5)

w_qtile = build_weights_quantile(zs_main, q=0.10)
bt_qtile = backtest(w_qtile, lr_main, cost_bps=5)

reversal_bt = {}
for m in REVERSAL_M_GRID:
    w_rev = build_cross_sectional_reversal(lr_main, m=m, c=1.0, G=1.0, w_max=0.05)
    reversal_bt[m] = backtest(w_rev, lr_main, cost_bps=5)

random_sharpes = []
for seed in range(RANDOM_SEEDS):
    w_rand = build_random_long_short_weights(lr_main, seed=seed)
    bt_rand = backtest(w_rand, lr_main, cost_bps=5)
    random_sharpes.append(perf_metrics(bt_rand["net"])["Sharpe"])

baseline_rows = {
    "Main (V1 equal-weight, 5 bps)": perf_metrics(bt_main["net"]),
    "Quantile (top/bot 10%, 5 bps)": perf_metrics(bt_qtile["net"]),
}
for m, bt in reversal_bt.items():
    baseline_rows[f"Cross-sectional reversal ({m}d, 5 bps)"] = perf_metrics(bt["net"])

baseline_rows["Random L-S (mean Sharpe, 5 bps)"] = {
    "Ann. Return": np.nan,
    "Ann. Vol": np.nan,
    "Sharpe": float(np.mean(random_sharpes)),
    "Max Drawdown": np.nan,
    "Calmar": np.nan,
    "Hit Rate": np.nan,
    "N Days": np.nan,
}

baseline_df = pd.DataFrame(baseline_rows).T
baseline_df.to_csv(f"{TABLES_DIR}/baseline_comparison{MAIN_SUFFIX}.csv")
format_results(baseline_df)

,Ann. Return,Ann. Vol,Sharpe,Max Drawdown,Calmar,Hit Rate,N Days
"Main (V1 equal-weight, 5 bps)",-20.87%,3.27%,-6.38,-96.42%,-0.22,31.8%,3584
"Quantile (top/bot 10%, 5 bps)",-22.02%,4.01%,-5.48,-97.10%,-0.23,33.0%,3584
"Cross-sectional reversal (1d, 5 bps)",-19.13%,8.14%,-2.35,-95.21%,-0.20,41.7%,3584
"Cross-sectional reversal (5d, 5 bps)",-12.31%,8.42%,-1.46,-85.51%,-0.14,44.7%,3584
"Cross-sectional reversal (10d, 5 bps)",-10.01%,8.49%,-1.18,-79.61%,-0.13,46.0%,3584
"Cross-sectional reversal (20d, 5 bps)",-7.36%,8.48%,-0.87,-71.24%,-0.10,47.0%,3584
"Random L-S (mean Sharpe, 5 bps)",--,--,-7.24,--,--,--,--


## 5. Factor-Neutral V3 Diagnostic

In [27]:
rolling_loadings = load_rolling_loadings(MAIN_SUFFIX)
if rolling_loadings is None:
    print(f"Rolling loadings not found for {MAIN_SUFFIX}. Re-run notebook 2.0.")
else:
    weights_v3_safe = neutralize_weights_over_time(weights_main, rolling_loadings, G=1.0, w_max=0.05, cond_max=1e8)
    bt_v3_safe_0 = backtest(weights_v3_safe, lr_main, cost_bps=0)
    bt_v3_safe_5 = backtest(weights_v3_safe, lr_main, cost_bps=5)

    v3_compare = pd.DataFrame({
        "V1 0bps": perf_metrics(backtest(weights_main, lr_main, cost_bps=0)["net"]),
        "V1 5bps": perf_metrics(bt_main["net"]),
        "V3 safe 0bps": perf_metrics(bt_v3_safe_0["net"]),
        "V3 safe 5bps": perf_metrics(bt_v3_safe_5["net"]),
    }).T
    v3_compare.to_csv(f"{TABLES_DIR}/v3_safe_comparison{MAIN_SUFFIX}.csv")

    exp_v1 = compute_factor_exposures(weights_main, rolling_loadings).abs().mean()
    exp_v3 = compute_factor_exposures(weights_v3_safe, rolling_loadings).abs().mean()
    exposure_compare = pd.DataFrame({"V1 abs mean exposure": exp_v1, "V3 safe abs mean exposure": exp_v3})
    exposure_compare.to_csv(f"{TABLES_DIR}/v3_exposure_comparison{MAIN_SUFFIX}.csv")

    print(v3_compare.to_string())
    print()
    print("Average absolute factor exposures:")
    print(exposure_compare.to_string())

              Ann. Return  Ann. Vol    Sharpe  Max Drawdown    Calmar  Hit Rate  N Days
V1 0bps         -0.018527  0.032709 -0.566421     -0.311645 -0.059450  0.481027  3584.0
V1 5bps         -0.208704  0.032716 -6.379310     -0.964200 -0.216453  0.318359  3584.0
V3 safe 0bps    -0.015774  0.027579 -0.571975     -0.270426 -0.058332  0.484933  3584.0
V3 safe 5bps    -0.197943  0.027580 -7.177116     -0.956627 -0.206918  0.293806  3584.0

Average absolute factor exposures:
      V1 abs mean exposure  V3 safe abs mean exposure
PC1               0.002080                   0.001545
PC2               0.004574                   0.000192
PC3               0.004628                   0.000136
PC4               0.004648                   0.000105
PC5               0.004670                   0.000102
PC6               0.004692                   0.000075
PC7               0.004609                   0.000062
PC8               0.004669                   0.000067
PC9               0.004713            

## 6. Save Summary Outputs

In [28]:
saved = sorted(os.listdir(TABLES_DIR))
print("Saved tables:")
for name in saved:
    if MAIN_SUFFIX.replace("_", "") in name.replace("_", "") or name.startswith("robustness_grid_reduced"):
        print(f"  {TABLES_DIR}/{name}")

Saved tables:
  ../reports/tables/baseline_comparison_L126_K10_M20.csv
  ../reports/tables/cost_sensitivity_L126_K10_M20.csv
  ../reports/tables/holding_period_sweep_L126_K10_M20.csv
  ../reports/tables/robustness_grid_L126_K10_M20.csv
  ../reports/tables/robustness_grid_reduced_L126_K10_M20.csv
  ../reports/tables/signal_direction_L126_K10_M20.csv
  ../reports/tables/threshold_sweep_L126_K10_M20.csv
  ../reports/tables/turnover_controls_L126_K10_M20.csv
  ../reports/tables/v3_exposure_comparison_L126_K10_M20.csv
  ../reports/tables/v3_safe_comparison_L126_K10_M20.csv
